<a href="https://colab.research.google.com/github/Jai-Kulkarni1905/Applied_Search_Intelligence/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jai-Kulkarni1905/Applied_Search_Intelligence/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

Prioritize the pages that have **meaningful search visibility** but are receiving **fewer clicks than expected for their average search position**.

The baseline will rank pages using two observable signals:

* **CTR relative to position** — identifies pages that appear to under-capture clicks compared with other pages at similar positions.
* **Impressions** — ensures that the opportunity is based on meaningful search visibility rather than a small amount of noisy data.

The output is a **review-priority ranking**, not a prediction that changing the page will improve its performance.

### Reason codes

* `low_ctr_visible_page` — the page has meaningful search visibility but its CTR is weak relative to pages at a similar search position.
* `insufficient_visibility` — the page does not have enough impressions for a reliable CTR-based review.
* `position_unavailable` — average position is unavailable or zero, so a position-adjusted CTR comparison cannot be made.
* `not_prioritized` — the page does not meet the conditions for the baseline review queue.

### Signal check 1 — CTR relative to position

**Signal:** CTR compared within average-position groups.

CTR should not be judged against one global threshold because pages ranking near position 1 naturally receive different click rates from pages ranking near position 10. We thus compare CTR within position tiers and check whether lower relative CTR is associated with the observed outcome.=

**Verdict:** based on the observed bucket table produced in the audit.

### Signal check 2 — Search visibility

**Signal:** GSC impressions.

Impressions provide the visibility context for the CTR signal. A low CTR based on very few impressions may simply be noise, so the rule should give greater priority to pages with meaningful existing visibility.


In [2]:
# --- Setup: connect DuckDB to the Hugging Face warehouse ---

import os
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata

# Read the HF token from Colab Secrets
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.sql("INSTALL httpfs; LOAD httpfs;")

# Register the token as a DuckDB secret
con.sql(f"""
CREATE OR REPLACE SECRET hf_token (
    TYPE huggingface,
    TOKEN '{os.environ["HF_TOKEN"]}'
)
""")

WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"

FACT_TABLE_GLOB = (
    f"{WAREHOUSE}/fact_content_daily_performance/**/*.parquet"
)

print("DuckDB connected and Hugging Face secret registered.")

DuckDB connected and Hugging Face secret registered.


In [3]:
march_path = (
    f"{WAREHOUSE}/fact_content_daily_performance/"
    f"month=2026-03/*.parquet"
)

print(march_path)

hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet


In [ ]:
march_data = con.sql(f"""SELECT * FROM read_parquet('{march_path}')""").df()

print("Rows:", len(march_data))
print("Columns:", len(march_data.columns))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 9841378
Columns: 31


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:

df = con.sql(f"""
    SELECT *
    FROM read_parquet('{march_path}')
    WHERE gsc_data_available IS TRUE
""").df()

df["report_date"] = pd.to_datetime(df["report_date"])

print(f"Total rows loaded: {len(df):,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total rows loaded: 3,611,061


2 time windows

In [5]:
feature_df = df[df["report_date"].dt.day <= 15].copy()

outcome_df = df[df["report_date"].dt.day >= 16].copy()

print("Information-window rows:", len(feature_df))
print("Outcome-window rows:", len(outcome_df))

Information-window rows: 1640237
Outcome-window rows: 1970824


page level feature table

In [34]:
features = (
    feature_df
    .groupby(["client_hash_id", "content_hash_id"])
    .agg(
        impressions=("gsc_impressions", "sum"),
        avg_position=("gsc_avg_position", "mean"),
        clicks=("gsc_clicks", "sum")
    )
    .reset_index()
)

features["ctr"] = features["clicks"] / features["impressions"].replace(0, np.nan)
features

,client_hash_id,content_hash_id,impressions,avg_position,clicks,ctr
0,client_0797ff3a1fc9a6a5,content_04c67f3541177192,119,12.639599,1,0.008403
1,client_0797ff3a1fc9a6a5,content_05acc92c165f4386,33,9.225529,0,0.000000
2,client_0797ff3a1fc9a6a5,content_0f30e04e709c7b5d,72,8.094074,0,0.000000
3,client_0797ff3a1fc9a6a5,content_1207efddce873942,283,12.587226,0,0.000000
4,client_0797ff3a1fc9a6a5,content_14df4b67b008d942,9,11.500000,0,0.000000
...,...,...,...,...,...,...
151976,client_ff644d8251367cbb,content_ffcb22817bdb0165,303,3.763376,0,0.000000
151977,client_ff644d8251367cbb,content_ffdaee2300ce7826,116,7.931378,0,0.000000
151978,client_ff644d8251367cbb,content_ffdeaa27dc03246b,814,16.119873,5,0.006143
151979,client_ff644d8251367cbb,content_ffdfde2be7d4f7d9,176,33.576000,0,0.000000


signal check 1

In [41]:

# --- CTR vs position (decision window only) ---
features = features[features["impressions"] >0]
position_bins = [0, 5, 10, 20, np.inf]
position_labels = ["1-5", "6-10", "11-20", "21+"]
features["position_bucket"] = pd.cut(features["avg_position"], bins=position_bins, labels=position_labels)

ctr_by_position = (
    features.groupby("position_bucket", observed=True)
    .agg(n=("ctr", "size"), mean_ctr=("ctr", "mean"), median_ctr=("ctr", "median"))
    .reset_index()
)
print("CTR by position bucket (n printed per bucket):\n", ctr_by_position)

CTR by position bucket (n printed per bucket):
   position_bucket      n  mean_ctr  median_ctr
0             1-5  39626  0.007392         0.0
1            6-10  46626  0.003952         0.0
2           11-20  26912  0.003337         0.0
3             21+  37511  0.002096         0.0


In [42]:
expected_ctr = (
    features
    .groupby("position_bucket", observed=False)["ctr"]
    .mean()
    .rename("expected_ctr")
)

features = features.merge(
    expected_ctr,
    on="position_bucket",
    how="left"
)

features["ctr_gap"] = (
    features["expected_ctr"] - features["ctr"]
)

features["relative_ctr"] = (
    features["ctr"] / features["expected_ctr"]
)

In [43]:
features.head()

,client_hash_id,content_hash_id,impressions,avg_position,clicks,ctr,position_bucket,expected_ctr,ctr_gap,relative_ctr
0,client_0797ff3a1fc9a6a5,content_04c67f3541177192,119,12.639599,1,0.008403,11-20,0.003337,-0.005066,2.518156
1,client_0797ff3a1fc9a6a5,content_05acc92c165f4386,33,9.225529,0,0.000000,6-10,0.003952,0.003952,0.000000
2,client_0797ff3a1fc9a6a5,content_0f30e04e709c7b5d,72,8.094074,0,0.000000,6-10,0.003952,0.003952,0.000000
3,client_0797ff3a1fc9a6a5,content_1207efddce873942,283,12.587226,0,0.000000,11-20,0.003337,0.003337,0.000000
4,client_0797ff3a1fc9a6a5,content_14df4b67b008d942,9,11.500000,0,0.000000,11-20,0.003337,0.003337,0.000000


signal check 2

In [32]:
visibility_check = (
    features[features["impressions"] > 300]
    .assign(
        visibility_bucket=lambda x: pd.cut(
            x["impressions"],
            bins=[0, 500, 1000, 2500, 5000, np.inf],
            labels=[
                "1-500",
                "501-1000",
                "1,001-2,500",
                "2,501-5,000",
                "5,000+"
            ]
        )
    )
    .groupby("visibility_bucket", observed=False)
    .agg(
        n=("content_hash_id", "size"),
        median_impressions=("impressions", "median")
    )
    .reset_index()
)

display(visibility_check)

,visibility_bucket,n,median_impressions
0,1-500,11136,387.0
1,501-1000,14799,713.0
2,"1,001-2,500",14699,1517.0
3,"2,501-5,000",6845,3371.0
4,"5,000+",5423,7975.0


building obbserved outcome

In [46]:
VISIBILITY_THRESHOLD = 500

# Start from the feature table already created above
baseline_df = features.copy()

# Position availability
baseline_df["position_available"] = (
    baseline_df["avg_position"].notna()
    & (baseline_df["avg_position"] > 0)
)

# Meaningful visibility
baseline_df["high_visibility"] = (
    baseline_df["impressions"] >= VISIBILITY_THRESHOLD
)

# Use the observed mean CTR for each position bucket
# CTR relative to the position-group expectation

In [58]:
baseline_df.head()

,client_hash_id,content_hash_id,impressions,avg_position,clicks,ctr,position_bucket,expected_ctr,ctr_gap,relative_ctr,position_available,high_visibility,reason_code,review_priority
0,client_0797ff3a1fc9a6a5,content_04c67f3541177192,119,12.639599,1,0.008403,11-20,0.003337,-0.005066,2.518156,True,False,insufficient_visibility,low
1,client_0797ff3a1fc9a6a5,content_05acc92c165f4386,33,9.225529,0,0.000000,6-10,0.003952,0.003952,0.000000,True,False,insufficient_visibility,low
2,client_0797ff3a1fc9a6a5,content_0f30e04e709c7b5d,72,8.094074,0,0.000000,6-10,0.003952,0.003952,0.000000,True,False,insufficient_visibility,low
3,client_0797ff3a1fc9a6a5,content_1207efddce873942,283,12.587226,0,0.000000,11-20,0.003337,0.003337,0.000000,True,False,insufficient_visibility,low
4,client_0797ff3a1fc9a6a5,content_14df4b67b008d942,9,11.500000,0,0.000000,11-20,0.003337,0.003337,0.000000,True,False,insufficient_visibility,low


In [74]:
# Reason codes
baseline_df["reason_code"] = np.select(
    [
        ~baseline_df["position_available"],

        baseline_df["position_available"]
        & ~baseline_df["high_visibility"]
        & (baseline_df['position_bucket'].isin(["11-20", "21+"])),

        baseline_df["position_available"]
        & ~baseline_df["high_visibility"]
        & (baseline_df['position_bucket'].isin(["1-5", "6-10"])),

        baseline_df["position_available"]
        & baseline_df["high_visibility"]
        & (baseline_df["ctr"] < baseline_df['expected_ctr']),

        baseline_df["position_available"]
        & baseline_df["high_visibility"]
        & (baseline_df["ctr"] >= baseline_df['expected_ctr']),

        baseline_df["position_available"]
        & ~baseline_df["high_visibility"]
        & (baseline_df["ctr"] >= baseline_df['expected_ctr'])
        & (baseline_df['position_bucket'].isin(["11-20", "21+"])),
    ],
    [
        "position_unavailable",
        "poor_position",
        "insufficient_visibility",
        "low_ctr_visible_page",
        "high_ctr_visible_page",
        "high_ctr_poor_position"
    ],
    default="not_prioritized"
)


In [75]:
# Review importance
# ------------------------------------------------------------
baseline_df["review_priority"] = np.select(
    [
        baseline_df["reason_code"] == "low_ctr_visible_page",
        baseline_df["reason_code"] == "insufficient_visibility",
        baseline_df["reason_code"] == "high_ctr_visible_page",
        baseline_df["reason_code"] == "position_unavailable",
        baseline_df["reason_code"] == "poor_position",
    ],
    ["med","med", "na", "high","high"],default="na"
)


In [76]:
baseline_df.reason_code.value_counts()

,count
reason_code,
insufficient_visibility,58860
poor_position,49999
low_ctr_visible_page,32080
high_ctr_visible_page,9736
position_unavailable,1306


In [77]:
# Display the resulting classification
# ------------------------------------------------------------
print("Reason-code counts:")
display(
    baseline_df["reason_code"]
    .value_counts()
    .rename_axis("reason_code")
    .reset_index(name="n")
)

print("\nReview-priority counts:")
display(
    baseline_df["review_priority"]
    .value_counts()
    .rename_axis("review_priority")
    .reset_index(name="n")
)

display(
    baseline_df[
        [
            "client_hash_id",
            "content_hash_id",
            "impressions",
            "ctr",
            "avg_position",
            "position_bucket",
            "expected_ctr",
            "relative_ctr",
            "reason_code",
            "review_priority"
        ]
    ].head(20)
)

Reason-code counts:


,reason_code,n
0,insufficient_visibility,58860
1,poor_position,49999
2,low_ctr_visible_page,32080
3,high_ctr_visible_page,9736
4,position_unavailable,1306



Review-priority counts:


,review_priority,n
0,med,90940
1,high,51305
2,na,9736


,client_hash_id,content_hash_id,impressions,ctr,avg_position,position_bucket,expected_ctr,relative_ctr,reason_code,review_priority
0,client_0797ff3a1fc9a6a5,content_04c67f3541177192,119,0.008403,12.639599,11-20,0.003337,2.518156,poor_position,high
1,client_0797ff3a1fc9a6a5,content_05acc92c165f4386,33,0.000000,9.225529,6-10,0.003952,0.000000,insufficient_visibility,med
2,client_0797ff3a1fc9a6a5,content_0f30e04e709c7b5d,72,0.000000,8.094074,6-10,0.003952,0.000000,insufficient_visibility,med
3,client_0797ff3a1fc9a6a5,content_1207efddce873942,283,0.000000,12.587226,11-20,0.003337,0.000000,poor_position,high
4,client_0797ff3a1fc9a6a5,content_14df4b67b008d942,9,0.000000,11.500000,11-20,0.003337,0.000000,poor_position,high
5,client_0797ff3a1fc9a6a5,content_167472cd0802a8f3,66,0.000000,12.282875,11-20,0.003337,0.000000,poor_position,high
6,client_0797ff3a1fc9a6a5,content_1fea2f270f3c1350,2,0.000000,3.500000,1-5,0.007392,0.000000,insufficient_visibility,med
7,client_0797ff3a1fc9a6a5,content_27f8100281413b37,11,0.000000,8.083333,6-10,0.003952,0.000000,insufficient_visibility,med
8,client_0797ff3a1fc9a6a5,content_2959111291cf661f,2,0.000000,28.500000,21+,0.002096,0.000000,poor_position,high
9,client_0797ff3a1fc9a6a5,content_2f719399052f18fc,8,0.000000,21.250000,21+,0.002096,0.000000,poor_position,high


In [ ]:
# ============================================================
# ACTION SCORE + ACTION PRIORITY
# Based on:
#   1. CTR relative to mean CTR for position bucket
#   2. Search visibility (impressions)
#   3. Existing reason-code classification
# ============================================================

# CTR shortfall relative to the position-bucket mean.
# Positive = below expected CTR.
# Negative = above expected CTR.
baseline_df["ctr_gap"] = (
    baseline_df["expected_ctr"] - baseline_df["ctr"]
)

# Convert the CTR gap into a relative shortfall.
# Example:
# expected CTR = 0.10, actual CTR = 0.05
# relative shortfall = 0.50 (50% below expected)
baseline_df["ctr_shortfall"] = np.where(
    baseline_df["expected_ctr"] > 0,
    (
        baseline_df["expected_ctr"] - baseline_df["ctr"]
    ) / baseline_df["expected_ctr"],
    0
)

# Only positive shortfalls represent under-capture.
baseline_df["ctr_shortfall"] = (
    baseline_df["ctr_shortfall"].clip(lower=0)
)

In [ ]:
first_half_impressions = (
    feature_df
    .groupby(["client_hash_id", "content_hash_id"], as_index=False)
    .agg(
        first_half_impressions=("gsc_impressions", "sum")
    )
)

second_half_impressions = (
    outcome_df
    .groupby(["client_hash_id", "content_hash_id"], as_index=False)
    .agg(
        second_half_impressions=("gsc_impressions", "sum")
    )
)

outcomes = first_half_impressions.merge(
    second_half_impressions,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

outcomes["is_declining"] = (
    outcomes["second_half_impressions"]
    < outcomes["first_half_impressions"]
)

display(outcomes.head())

ctr signal

In [ ]:
ctr_audit = signal_df.merge(
    outcomes[
        [
            "client_hash_id",
            "content_hash_id",
            "is_declining"
        ]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

ctr_audit["low_relative_ctr"] = (
    ctr_audit["relative_ctr"] < 1
)

ctr_verdict_table = (
    ctr_audit
    .groupby("position_bucket", observed=False)
    .agg(
        n=("content_hash_id", "size"),
        decline_rate=("is_declining", "mean"),
        median_relative_ctr=("relative_ctr", "median")
    )
    .reset_index()
)

display(ctr_verdict_table)

visibility signal

In [ ]:
visibility_audit = features.merge(
    outcomes[
        [
            "client_hash_id",
            "content_hash_id",
            "is_declining"
        ]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

visibility_audit["visibility_bucket"] = pd.cut(
    visibility_audit["impressions"],
    bins=[0, 100, 500, 1000, 5000, np.inf],
    labels=[
        "1-100",
        "101-500",
        "501-1,000",
        "1,001-5,000",
        "5,000+"
    ]
)

visibility_verdict_table = (
    visibility_audit
    .groupby("visibility_bucket", observed=False)
    .agg(
        n=("content_hash_id", "size"),
        decline_rate=("is_declining", "mean")
    )
    .reset_index()
)

display(visibility_verdict_table)

baseline score


In [ ]:
scoring = signal_df.copy()

# Pages with enough observable visibility are eligible.
MIN_IMPRESSIONS = 100

scoring["eligible"] = (
    (scoring["impressions"] >= MIN_IMPRESSIONS) &
    (scoring["expected_ctr"] > 0)
)

scoring = scoring[scoring["eligible"]].copy()

# Relative CTR shortfall.
scoring["ctr_shortfall"] = (
    1 - scoring["relative_ctr"]
).clip(lower=0)

# Visibility-weighted opportunity score.
scoring["score"] = (
    np.log1p(scoring["impressions"])
    * scoring["ctr_shortfall"]
)

reason code and action

In [ ]:
scoring["reason_code"] = "low_ctr_visible_page"
scoring["action"] = "review_ctr"

ranking

In [ ]:
queue = (
    scoring
    .sort_values(
        ["score", "impressions"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

queue.insert(
    0,
    "rank",
    np.arange(1, len(queue) + 1)
)

display(
    queue[
        [
            "rank",
            "client_hash_id",
            "content_hash_id",
            "score",
            "reason_code",
            "action",
            "impressions",
            "clicks",
            "ctr",
            "avg_position",
            "expected_ctr",
            "ctr_shortfall"
        ]
    ].head(10)
)

precision at k

In [ ]:
evaluation = queue.merge(
    outcomes[
        [
            "client_hash_id",
            "content_hash_id",
            "is_declining"
        ]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

base_rate = evaluation["is_declining"].mean()

print(f"Observed decline base rate: {base_rate:.3f}")

In [ ]:
for k in [10, 50, 100]:
    top_k = evaluation.head(k)

    precision_at_k = top_k["is_declining"].mean()

    print(
        f"Precision@{k}: "
        f"{precision_at_k:.3f} "
        f"({top_k['is_declining'].sum()}/{len(top_k)})"
    )

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.